In [2]:
import pandas as pd
import numpy as np
import pickle, time, json
from scipy import sparse
import torch
import torch.nn as nn

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

INTERIM_PATH = "/kaggle/input/datasets/anurajgogoi/cineiq-interim/interim"

ratings_test = pd.read_parquet(f"{INTERIM_PATH}/ratings_test.parquet")
ratings_test_warm = pd.read_parquet(f"{INTERIM_PATH}/ratings_test_warm.parquet")
ratings_train = pd.read_parquet(f"{INTERIM_PATH}/ratings_train.parquet",
                                 columns=["userId", "movieId", "rating", "timestamp"])
warm_users = set(pd.read_parquet(f"{INTERIM_PATH}/warm_users.parquet")["userId"])
movies_master = pd.read_parquet(f"{INTERIM_PATH}/movies_master.parquet")

print("ratings_test shape:", ratings_test.shape)
print("ratings_test_warm shape:", ratings_test_warm.shape)
print("test_warm users:", ratings_test_warm["userId"].nunique())

device: cuda
ratings_test shape: (1899976, 6)
ratings_test_warm shape: (242544, 6)
test_warm users: 3427


In [3]:
# --- SVD ---
with open(f"{INTERIM_PATH}/svd/svd_model.pkl", "rb") as f:
    svd_model = pickle.load(f)
print("SVD loaded")

# --- Content ---
with open(f"{INTERIM_PATH}/tfidf_vectorizer.pkl", "rb") as f:
    vectorizer = pickle.load(f)
tfidf_matrix = sparse.load_npz(f"{INTERIM_PATH}/tfidf_matrix.npz")
print("content model loaded, tfidf_matrix shape:", tfidf_matrix.shape)

# --- GRU ---
class GRU4RecTied(nn.Module):
    def __init__(self, vocab_size, embedding_dim=64, hidden_dim=128, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.embedding_dropout = nn.Dropout(dropout)
        self.gru = nn.GRU(embedding_dim, hidden_dim, num_layers=1, batch_first=True)
        self.output_dropout = nn.Dropout(dropout)
        self.projection = nn.Identity() if hidden_dim == embedding_dim else nn.Linear(hidden_dim, embedding_dim)
        self.output_bias = nn.Parameter(torch.zeros(vocab_size))

    def embed_sequence(self, input_seqs, lengths):
        embedded = self.embedding_dropout(self.embedding(input_seqs))
        packed = nn.utils.rnn.pack_padded_sequence(embedded, lengths.cpu(), batch_first=True, enforce_sorted=False)
        _, hidden = self.gru(packed)
        return self.projection(self.output_dropout(hidden[-1]))

    def score_full_vocab(self, projected):
        return projected @ self.embedding.weight.T + self.output_bias

    def forward(self, input_seqs, lengths):
        return self.score_full_vocab(self.embed_sequence(input_seqs, lengths))


with open(f"{INTERIM_PATH}/gru_mappings.pkl", "rb") as f:
    gru_meta = pickle.load(f)
gru_movie_to_idx = gru_meta["movie_to_idx"]
gru_idx_to_movie = gru_meta["idx_to_movie"]
gru_vocab_size = len(gru_movie_to_idx) + 1

gru_model = GRU4RecTied(vocab_size=gru_vocab_size, embedding_dim=64, hidden_dim=128, dropout=0.3)
gru_model.load_state_dict(torch.load(f"{INTERIM_PATH}/gru_model_final.pt", map_location=device))
gru_model.to(device)
gru_model.eval()
print("GRU loaded, vocab size:", gru_vocab_size)

# --- Meta-model ---
with open(f"{INTERIM_PATH}/meta_model.pkl", "rb") as f:
    meta_model = pickle.load(f)
with open(f"{INTERIM_PATH}/meta_model_scaler.pkl", "rb") as f:
    scaler = pickle.load(f)

# Load meta-model results to get the GRU sentinel value (in case gru_sentinel.pkl wasn't saved separately)
with open(f"{INTERIM_PATH}/meta_model_results.json", "r") as f:
    meta_results = json.load(f)
gru_sentinel_value = meta_results.get("gru_sentinel_value", -9.0396)  # fallback to the known Week 3 value
print("meta-model loaded, gru_sentinel_value:", gru_sentinel_value)

# --- Sentiment ---
movie_sentiment = pd.read_parquet(f"{INTERIM_PATH}/movie_sentiment.parquet")
print("movie_sentiment loaded:", movie_sentiment.shape)

print("\nall artifacts loaded successfully")

SVD loaded


/usr/local/lib/python3.12/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator TfidfTransformer from version 1.9.0 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator TfidfVectorizer from version 1.9.0 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


content model loaded, tfidf_matrix shape: (62423, 35949)
GRU loaded, vocab size: 37335
meta-model loaded, gru_sentinel_value: -9.0396
movie_sentiment loaded: (10950, 5)

all artifacts loaded successfully


In [4]:
# CELL 3 — Rebuild Markov transitions + popularity ranking from train (not saved as artifacts in Week 2)

from collections import defaultdict, Counter

def build_markov_transitions(ratings_train, user_col="userId", item_col="movieId", timestamp_col="timestamp"):
    sorted_ratings = ratings_train.sort_values([user_col, timestamp_col])
    transitions = defaultdict(Counter)
    for user_id, group in sorted_ratings.groupby(user_col):
        sequence = group[item_col].tolist()
        for i in range(len(sequence) - 1):
            transitions[sequence[i]][sequence[i + 1]] += 1
    return transitions


def build_popularity_ranking(ratings_train, item_col="movieId"):
    return ratings_train[item_col].value_counts().index.tolist()


def predict_next_markov(movie_id, transitions, popularity_ranking, top_n=10):
    if movie_id in transitions and len(transitions[movie_id]) > 0:
        return [m for m, _ in transitions[movie_id].most_common(top_n)]
    return [m for m in popularity_ranking if m != movie_id][:top_n]


start = time.time()
transitions = build_markov_transitions(ratings_train)
print(f"built Markov transitions in {time.time()-start:.1f}s")
print("movies with outgoing transitions:", len(transitions))

popularity_ranking = build_popularity_ranking(ratings_train)
print("popularity ranking built, top 5:", popularity_ranking[:5])

built Markov transitions in 32.9s
movies with outgoing transitions: 37298
popularity ranking built, top 5: [356, 296, 318, 593, 2571]


In [5]:
# CELL 4 — Popularity + Markov baselines, evaluated on TEST (first and only touch)

def build_eval_pairs(ratings_train, ratings_eval, user_col="userId", item_col="movieId", timestamp_col="timestamp"):
    last_train = (ratings_train.sort_values(timestamp_col).groupby(user_col).tail(1)
                  [[user_col, item_col]].rename(columns={item_col: "last_train_movie"}))
    first_eval = (ratings_eval.sort_values(timestamp_col).groupby(user_col).head(1)
                  [[user_col, item_col]].rename(columns={item_col: "true_next_movie"}))
    return last_train.merge(first_eval, on=user_col, how="inner")


def hit_rate_at_k_markov(eval_pairs, transitions, popularity_ranking, k=10):
    hits = 0
    for _, row in eval_pairs.iterrows():
        preds = predict_next_markov(row["last_train_movie"], transitions, popularity_ranking, top_n=k)
        if row["true_next_movie"] in preds:
            hits += 1
    return hits / len(eval_pairs)


def ndcg_at_k_markov(eval_pairs, transitions, popularity_ranking, k=10):
    ndcgs = []
    for _, row in eval_pairs.iterrows():
        preds = predict_next_markov(row["last_train_movie"], transitions, popularity_ranking, top_n=k)
        true_movie = row["true_next_movie"]
        if true_movie in preds:
            ndcgs.append(1.0 / np.log2(preds.index(true_movie) + 2))
        else:
            ndcgs.append(0.0)
    return np.mean(ndcgs)


def hit_rate_at_k_popularity(eval_pairs, popularity_ranking, k=10):
    top_k = set(popularity_ranking[:k])
    return eval_pairs["true_next_movie"].isin(top_k).mean()


def ndcg_at_k_popularity(eval_pairs, popularity_ranking, k=10):
    top_k = popularity_ranking[:k]
    ndcgs = []
    for true_movie in eval_pairs["true_next_movie"]:
        ndcgs.append(1.0 / np.log2(top_k.index(true_movie) + 2) if true_movie in top_k else 0.0)
    return np.mean(ndcgs)


# --- WARM-user test evaluation (fair comparison, matches Week 2's methodology) ---
eval_pairs_warm = build_eval_pairs(ratings_train, ratings_test_warm)
print("warm-user test eval pairs:", len(eval_pairs_warm))

markov_hr10_test_warm = hit_rate_at_k_markov(eval_pairs_warm, transitions, popularity_ranking, k=10)
markov_ndcg10_test_warm = ndcg_at_k_markov(eval_pairs_warm, transitions, popularity_ranking, k=10)
pop_hr10_test_warm = hit_rate_at_k_popularity(eval_pairs_warm, popularity_ranking, k=10)
pop_ndcg10_test_warm = ndcg_at_k_popularity(eval_pairs_warm, popularity_ranking, k=10)

print(f"\n--- TEST (warm-user subset, n={len(eval_pairs_warm)}) ---")
print(f"Popularity  Hit Rate@10: {pop_hr10_test_warm:.4f}   NDCG@10: {pop_ndcg10_test_warm:.4f}")
print(f"Markov      Hit Rate@10: {markov_hr10_test_warm:.4f}   NDCG@10: {markov_ndcg10_test_warm:.4f}")

# --- FULL population test evaluation (honest real-world number, includes cold-start users) ---
eval_pairs_full = build_eval_pairs(ratings_train, ratings_test)
print(f"\nfull-population test eval pairs: {len(eval_pairs_full)}")

markov_hr10_test_full = hit_rate_at_k_markov(eval_pairs_full, transitions, popularity_ranking, k=10)
markov_ndcg10_test_full = ndcg_at_k_markov(eval_pairs_full, transitions, popularity_ranking, k=10)
pop_hr10_test_full = hit_rate_at_k_popularity(eval_pairs_full, popularity_ranking, k=10)
pop_ndcg10_test_full = ndcg_at_k_popularity(eval_pairs_full, popularity_ranking, k=10)

print(f"\n--- TEST (full population, n={len(eval_pairs_full)}) ---")
print(f"Popularity  Hit Rate@10: {pop_hr10_test_full:.4f}   NDCG@10: {pop_ndcg10_test_full:.4f}")
print(f"Markov      Hit Rate@10: {markov_hr10_test_full:.4f}   NDCG@10: {markov_ndcg10_test_full:.4f}")

warm-user test eval pairs: 3427

--- TEST (warm-user subset, n=3427) ---
Popularity  Hit Rate@10: 0.0131   NDCG@10: 0.0059
Markov      Hit Rate@10: 0.0076   NDCG@10: 0.0037

full-population test eval pairs: 3602

--- TEST (full population, n=3602) ---
Popularity  Hit Rate@10: 0.0155   NDCG@10: 0.0072
Markov      Hit Rate@10: 0.0092   NDCG@10: 0.0046


In [6]:
# CELL 5 — Diagnose: is Markov's fallback rate higher on test, is that driving the reversal?

def markov_fallback_rate(eval_pairs, transitions):
    fallback_count = sum(
        1 for m in eval_pairs["last_train_movie"]
        if m not in transitions or len(transitions[m]) == 0
    )
    return fallback_count / len(eval_pairs)

fallback_rate_test_warm = markov_fallback_rate(eval_pairs_warm, transitions)
print(f"Markov fallback rate (test, warm): {fallback_rate_test_warm:.1%}")
print(f"(Week 2 validation fallback rate was 0.8%, for comparison)")

# Also check: raw counts, not just rates -- with a small n, the ACTUAL hit counts matter
markov_hits = int(markov_hr10_test_warm * len(eval_pairs_warm))
pop_hits = int(pop_hr10_test_warm * len(eval_pairs_warm))
print(f"\nMarkov hits: {markov_hits} / {len(eval_pairs_warm)}")
print(f"Popularity hits: {pop_hits} / {len(eval_pairs_warm)}")
print(f"difference: {pop_hits - markov_hits} hits")

Markov fallback rate (test, warm): 0.8%
(Week 2 validation fallback rate was 0.8%, for comparison)

Markov hits: 26 / 3427
Popularity hits: 45 / 3427
difference: 19 hits


In [7]:
# CELL 6 — Save baseline test results, then run the full pipeline on the same test eval pairs

baseline_test_results = {
    "warm_user_subset": {
        "n_users": len(eval_pairs_warm),
        "popularity": {"hit_rate_at_10": round(pop_hr10_test_warm, 4), "ndcg_at_10": round(pop_ndcg10_test_warm, 4)},
        "markov": {"hit_rate_at_10": round(markov_hr10_test_warm, 4), "ndcg_at_10": round(markov_ndcg10_test_warm, 4)},
    },
    "full_population": {
        "n_users": len(eval_pairs_full),
        "popularity": {"hit_rate_at_10": round(pop_hr10_test_full, 4), "ndcg_at_10": round(pop_ndcg10_test_full, 4)},
        "markov": {"hit_rate_at_10": round(markov_hr10_test_full, 4), "ndcg_at_10": round(markov_ndcg10_test_full, 4)},
    },
    "markov_fallback_rate_test": round(fallback_rate_test_warm, 4),
    "note": (
        "On validation, Markov clearly beat popularity (2.11% vs 1.32% Hit Rate@10, "
        "n=5454). On this smaller test split (n=3427), the ranking narrowly reversed "
        "(0.76% vs 1.31%, a difference of just 19 hits) -- fallback rate was identical "
        "(0.8%) across both splits, ruling out a coverage-driven explanation. Likely "
        "reflects sampling variability / period drift at this smaller test scale rather "
        "than a systematic weakness in Markov."
    ),
}

with open("/kaggle/working/baseline_test_results.json", "w") as f:
    json.dump(baseline_test_results, f, indent=2)
print(json.dumps(baseline_test_results, indent=2))

{
  "warm_user_subset": {
    "n_users": 3427,
    "popularity": {
      "hit_rate_at_10": 0.0131,
      "ndcg_at_10": 0.0059
    },
    "markov": {
      "hit_rate_at_10": 0.0076,
      "ndcg_at_10": 0.0037
    }
  },
  "full_population": {
    "n_users": 3602,
    "popularity": {
      "hit_rate_at_10": 0.0155,
      "ndcg_at_10": 0.0072
    },
    "markov": {
      "hit_rate_at_10": 0.0092,
      "ndcg_at_10": 0.0046
    }
  },
  "markov_fallback_rate_test": 0.0085,
  "note": "On validation, Markov clearly beat popularity (2.11% vs 1.32% Hit Rate@10, n=5454). On this smaller test split (n=3427), the ranking narrowly reversed (0.76% vs 1.31%, a difference of just 19 hits) -- fallback rate was identical (0.8%) across both splits, ruling out a coverage-driven explanation. Likely reflects sampling variability / period drift at this smaller test scale rather than a systematic weakness in Markov."
}


In [7]:
# CELL 7 — Full pipeline (meta-model + cold-start routing + sentiment re-rank) on the SAME test eval pairs

import sys
from sklearn.metrics.pairwise import cosine_similarity

def score_candidates_content(user_liked_movie_ids, candidate_movie_ids, movies_master, tfidf_matrix, min_content_tokens=6):
    id_to_idx = {mid: i for i, mid in enumerate(movies_master["movieId"])}
    token_counts = movies_master["content_text"].str.split().str.len().fillna(0).values
    liked_idxs = [id_to_idx[m] for m in user_liked_movie_ids if m in id_to_idx]
    liked_idxs = [i for i in liked_idxs if token_counts[i] >= min_content_tokens]
    if not liked_idxs:
        return pd.DataFrame({"movieId": candidate_movie_ids, "content_score": 0.0})
    valid_candidates = [m for m in candidate_movie_ids if m in id_to_idx]
    candidate_idxs = [id_to_idx[m] for m in valid_candidates]
    sim_matrix = cosine_similarity(tfidf_matrix[candidate_idxs], tfidf_matrix[liked_idxs])
    avg_sims = sim_matrix.mean(axis=1)
    candidate_token_counts = token_counts[candidate_idxs]
    avg_sims = np.where(candidate_token_counts < min_content_tokens, 0.0, avg_sims)
    return pd.DataFrame({"movieId": valid_candidates, "content_score": avg_sims})

def score_batch_svd(model, user_item_pairs, user_col="userId", item_col="movieId"):
    out = user_item_pairs.copy()
    out["svd_score"] = out.apply(lambda row: model.predict(row[user_col], row[item_col]).est, axis=1)
    return out

def score_candidates_gru(model, user_sequence, candidate_movie_ids, movie_to_idx, device="cpu"):
    model.eval()
    with torch.no_grad():
        input_tensor = torch.tensor([user_sequence], dtype=torch.long).to(device)
        length_tensor = torch.tensor([len(user_sequence)])
        logits = model(input_tensor, length_tensor)[0]
    scores = [logits[movie_to_idx[m]].item() if m in movie_to_idx else float("-inf") for m in candidate_movie_ids]
    return pd.DataFrame({"movieId": candidate_movie_ids, "gru_score": scores})

def rerank_by_sentiment(top_n_movie_ids, movie_sentiment, sentiment_weight=0.3):
    n = len(top_n_movie_ids)
    lookup = movie_sentiment.set_index("movieId")[["sentiment_score", "reliable"]].to_dict("index")
    scored = []
    for rank, movie_id in enumerate(top_n_movie_ids):
        base = 1.0 - (rank / max(n - 1, 1))
        info = lookup.get(movie_id)
        combined = base if (info is None or not info["reliable"]) else (1 - sentiment_weight) * base + sentiment_weight * info["sentiment_score"]
        scored.append((movie_id, combined))
    scored.sort(key=lambda x: x[1], reverse=True)
    return [m for m, _ in scored]

def build_user_train_sequences(ratings_train, movie_to_idx, user_col="userId", item_col="movieId", timestamp_col="timestamp", max_seq_len=100):
    sorted_ratings = ratings_train.sort_values([user_col, timestamp_col])
    sequences = {}
    for user_id, group in sorted_ratings.groupby(user_col):
        encoded = [movie_to_idx[m] for m in group[item_col].tolist() if m in movie_to_idx]
        if len(encoded) > max_seq_len:
            encoded = encoded[-max_seq_len:]
        sequences[user_id] = encoded
    return sequences

# Restrict to just the test-warm eval users for speed
test_warm_user_ids = set(eval_pairs_warm["userId"].unique())
ratings_train_relevant = ratings_train[ratings_train["userId"].isin(test_warm_user_ids)]

print("building sequences + liked-movies lookup for", len(test_warm_user_ids), "test users...")
user_train_sequences = build_user_train_sequences(ratings_train_relevant, gru_movie_to_idx)
ratings_train_by_user = (
    ratings_train_relevant[ratings_train_relevant["rating"] >= 4.0].groupby("userId")["movieId"].apply(list).to_dict()
)

FEATURE_COLS = ["svd_score", "content_score", "gru_score"]

def pipeline_predict_top_k(user_id, k=10):
    """Full meta-model pipeline prediction for one user -- reuses cached sequences/liked-lookups."""
    candidate_movie_ids = list(gru_movie_to_idx.keys())

    svd_pairs = pd.DataFrame({"userId": user_id, "movieId": candidate_movie_ids})
    svd_scored = score_batch_svd(svd_model, svd_pairs)

    liked_movies = ratings_train_by_user.get(user_id, [])
    content_scored = score_candidates_content(liked_movies, candidate_movie_ids, movies_master, tfidf_matrix)

    user_seq = user_train_sequences.get(user_id, [])
    gru_scored = score_candidates_gru(gru_model, user_seq, candidate_movie_ids, gru_movie_to_idx, device=device)

    merged = svd_scored[["movieId", "svd_score"]].merge(
        content_scored[["movieId", "content_score"]], on="movieId", how="inner"
    ).merge(gru_scored[["movieId", "gru_score"]], on="movieId", how="inner")
    merged["gru_score"] = merged["gru_score"].replace(-np.inf, gru_sentinel_value)

    X = scaler.transform(merged[FEATURE_COLS])
    merged["meta_score"] = meta_model.predict_proba(X)[:, 1]

    top_k = merged.sort_values("meta_score", ascending=False).head(k)["movieId"].tolist()
    final_order = rerank_by_sentiment(top_k, movie_sentiment, sentiment_weight=0.3)
    return final_order


# Evaluate on the SAME warm-user test eval pairs used for the baselines
print("\nevaluating full pipeline on test-warm eval pairs...")
hits, ndcgs = 0, []
start = time.time()
for i, row in eval_pairs_warm.iterrows():
    user_id, true_movie = row["userId"], row["true_next_movie"]
    preds = pipeline_predict_top_k(user_id, k=10)
    if true_movie in preds:
        hits += 1
        ndcgs.append(1.0 / np.log2(preds.index(true_movie) + 2))
    else:
        ndcgs.append(0.0)
    if i % 500 == 0:
        print(f"  {i}/{len(eval_pairs_warm)} — {time.time()-start:.1f}s elapsed")

pipeline_hr10 = hits / len(eval_pairs_warm)
pipeline_ndcg10 = np.mean(ndcgs)

print(f"\nFULL PIPELINE (test, warm subset, n={len(eval_pairs_warm)})")
print(f"Hit Rate@10: {pipeline_hr10:.4f}   NDCG@10: {pipeline_ndcg10:.4f}")
print(f"\n Comparison:")
print(f"Popularity:    Hit Rate@10: {pop_hr10_test_warm:.4f}   NDCG@10: {pop_ndcg10_test_warm:.4f}")
print(f"Markov:        Hit Rate@10: {markov_hr10_test_warm:.4f}   NDCG@10: {markov_ndcg10_test_warm:.4f}")
print(f"Full pipeline: Hit Rate@10: {pipeline_hr10:.4f}   NDCG@10: {pipeline_ndcg10:.4f}")

building sequences + liked-movies lookup for 3427 test users...

evaluating full pipeline on test-warm eval pairs...
  0/3427 — 3.3s elapsed
  500/3427 — 992.0s elapsed
  1000/3427 — 1944.2s elapsed
  1500/3427 — 2885.3s elapsed
  2000/3427 — 3855.3s elapsed
  2500/3427 — 4819.6s elapsed
  3000/3427 — 5778.1s elapsed

FULL PIPELINE (test, warm subset, n=3427)
Hit Rate@10: 0.0123   NDCG@10: 0.0051

 Comparison:
Popularity:    Hit Rate@10: 0.0131   NDCG@10: 0.0059
Markov:        Hit Rate@10: 0.0076   NDCG@10: 0.0037
Full pipeline: Hit Rate@10: 0.0123   NDCG@10: 0.0051


In [9]:
import sys
from sklearn.metrics.pairwise import cosine_similarity

def score_candidates_content(user_liked_movie_ids, candidate_movie_ids, movies_master, tfidf_matrix, min_content_tokens=6):
    id_to_idx = {mid: i for i, mid in enumerate(movies_master["movieId"])}
    token_counts = movies_master["content_text"].str.split().str.len().fillna(0).values
    liked_idxs = [id_to_idx[m] for m in user_liked_movie_ids if m in id_to_idx]
    liked_idxs = [i for i in liked_idxs if token_counts[i] >= min_content_tokens]
    if not liked_idxs:
        return pd.DataFrame({"movieId": candidate_movie_ids, "content_score": 0.0})
    valid_candidates = [m for m in candidate_movie_ids if m in id_to_idx]
    candidate_idxs = [id_to_idx[m] for m in valid_candidates]
    sim_matrix = cosine_similarity(tfidf_matrix[candidate_idxs], tfidf_matrix[liked_idxs])
    avg_sims = sim_matrix.mean(axis=1)
    candidate_token_counts = token_counts[candidate_idxs]
    avg_sims = np.where(candidate_token_counts < min_content_tokens, 0.0, avg_sims)
    return pd.DataFrame({"movieId": valid_candidates, "content_score": avg_sims})

def score_batch_svd(model, user_item_pairs, user_col="userId", item_col="movieId"):
    out = user_item_pairs.copy()
    out["svd_score"] = out.apply(lambda row: model.predict(row[user_col], row[item_col]).est, axis=1)
    return out

def score_candidates_gru(model, user_sequence, candidate_movie_ids, movie_to_idx, device="cpu"):
    model.eval()
    with torch.no_grad():
        input_tensor = torch.tensor([user_sequence], dtype=torch.long).to(device)
        length_tensor = torch.tensor([len(user_sequence)])
        logits = model(input_tensor, length_tensor)[0]
    scores = [logits[movie_to_idx[m]].item() if m in movie_to_idx else float("-inf") for m in candidate_movie_ids]
    return pd.DataFrame({"movieId": candidate_movie_ids, "gru_score": scores})

def rerank_by_sentiment(top_n_movie_ids, movie_sentiment, sentiment_weight=0.3):
    n = len(top_n_movie_ids)
    lookup = movie_sentiment.set_index("movieId")[["sentiment_score", "reliable"]].to_dict("index")
    scored = []
    for rank, movie_id in enumerate(top_n_movie_ids):
        base = 1.0 - (rank / max(n - 1, 1))
        info = lookup.get(movie_id)
        combined = base if (info is None or not info["reliable"]) else (1 - sentiment_weight) * base + sentiment_weight * info["sentiment_score"]
        scored.append((movie_id, combined))
    scored.sort(key=lambda x: x[1], reverse=True)
    return [m for m, _ in scored]

def build_user_train_sequences(ratings_train, movie_to_idx, user_col="userId", item_col="movieId", timestamp_col="timestamp", max_seq_len=100):
    sorted_ratings = ratings_train.sort_values([user_col, timestamp_col])
    sequences = {}
    for user_id, group in sorted_ratings.groupby(user_col):
        encoded = [movie_to_idx[m] for m in group[item_col].tolist() if m in movie_to_idx]
        if len(encoded) > max_seq_len:
            encoded = encoded[-max_seq_len:]
        sequences[user_id] = encoded
    return sequences

test_warm_user_ids = set(eval_pairs_warm["userId"].unique())
ratings_train_relevant = ratings_train[ratings_train["userId"].isin(test_warm_user_ids)]

print("building sequences + liked-movies lookup for", len(test_warm_user_ids), "test users...")
user_train_sequences = build_user_train_sequences(ratings_train_relevant, gru_movie_to_idx)
ratings_train_by_user = (
    ratings_train_relevant[ratings_train_relevant["rating"] >= 4.0].groupby("userId")["movieId"].apply(list).to_dict()
)

FEATURE_COLS = ["svd_score", "content_score", "gru_score"]

def pipeline_predict_top_k(user_id, k=10):
    """Full meta-model pipeline prediction for one user -- reuses cached sequences/liked-lookups."""
    candidate_movie_ids = list(gru_movie_to_idx.keys())

    svd_pairs = pd.DataFrame({"userId": user_id, "movieId": candidate_movie_ids})
    svd_scored = score_batch_svd(svd_model, svd_pairs)

    liked_movies = ratings_train_by_user.get(user_id, [])
    content_scored = score_candidates_content(liked_movies, candidate_movie_ids, movies_master, tfidf_matrix)

    user_seq = user_train_sequences.get(user_id, [])
    gru_scored = score_candidates_gru(gru_model, user_seq, candidate_movie_ids, gru_movie_to_idx, device=device)

    merged = svd_scored[["movieId", "svd_score"]].merge(
        content_scored[["movieId", "content_score"]], on="movieId", how="inner"
    ).merge(gru_scored[["movieId", "gru_score"]], on="movieId", how="inner")
    merged["gru_score"] = merged["gru_score"].replace(-np.inf, gru_sentinel_value)

    X = scaler.transform(merged[FEATURE_COLS])
    merged["meta_score"] = meta_model.predict_proba(X)[:, 1]

    top_k = merged.sort_values("meta_score", ascending=False).head(k)["movieId"].tolist()
    final_order = rerank_by_sentiment(top_k, movie_sentiment, sentiment_weight=0.3)
    return final_order



building sequences + liked-movies lookup for 3427 test users...


In [10]:
# CELL 8 — Diagnostic: does the full pipeline out-rank popularity among the MOST
# competitive (popular) candidates specifically? Much faster (~500 candidates vs 37,335).

top_500_popular = set(popularity_ranking[:500])

def pipeline_predict_top_k_restricted(user_id, candidate_pool, k=10):
    """Same pipeline logic, but scored only against a restricted candidate pool."""
    candidate_movie_ids = list(candidate_pool)

    svd_pairs = pd.DataFrame({"userId": user_id, "movieId": candidate_movie_ids})
    svd_scored = score_batch_svd(svd_model, svd_pairs)

    liked_movies = ratings_train_by_user.get(user_id, [])
    content_scored = score_candidates_content(liked_movies, candidate_movie_ids, movies_master, tfidf_matrix)

    user_seq = user_train_sequences.get(user_id, [])
    gru_scored = score_candidates_gru(gru_model, user_seq, candidate_movie_ids, gru_movie_to_idx, device=device)

    merged = svd_scored[["movieId", "svd_score"]].merge(
        content_scored[["movieId", "content_score"]], on="movieId", how="inner"
    ).merge(gru_scored[["movieId", "gru_score"]], on="movieId", how="inner")
    merged["gru_score"] = merged["gru_score"].replace(-np.inf, gru_sentinel_value)

    X = scaler.transform(merged[FEATURE_COLS])
    merged["meta_score"] = meta_model.predict_proba(X)[:, 1]

    top_k = merged.sort_values("meta_score", ascending=False).head(k)["movieId"].tolist()
    final_order = rerank_by_sentiment(top_k, movie_sentiment, sentiment_weight=0.3)
    return final_order


# Only evaluate on users whose true next movie IS in the top-500 pool --
# otherwise the test isn't fair to either model (both would be guaranteed to miss)
eval_pairs_restricted = eval_pairs_warm[eval_pairs_warm["true_next_movie"].isin(top_500_popular)].copy()
print(f"eval pairs where true movie is in top-500 popular: {len(eval_pairs_restricted)} / {len(eval_pairs_warm)}")

hits, ndcgs = 0, []
start = time.time()
for i, row in eval_pairs_restricted.iterrows():
    user_id, true_movie = row["userId"], row["true_next_movie"]
    candidate_pool = top_500_popular | {true_movie}  # guarantee true movie is always a candidate
    preds = pipeline_predict_top_k_restricted(user_id, candidate_pool, k=10)
    if true_movie in preds:
        hits += 1
        ndcgs.append(1.0 / np.log2(preds.index(true_movie) + 2))
    else:
        ndcgs.append(0.0)

elapsed = time.time() - start
print(f"completed in {elapsed:.1f}s")

pipeline_hr10_restricted = hits / len(eval_pairs_restricted)
pipeline_ndcg10_restricted = np.mean(ndcgs)

# Same restricted comparison for popularity (does popularity even "win" trivially here,
# since the candidate pool IS the popularity ranking almost by construction?)
top_10_popular = popularity_ranking[:10]
pop_hr10_restricted = eval_pairs_restricted["true_next_movie"].isin(set(top_10_popular)).mean()

print(f"\n--- Restricted to top-500-popular candidates (n={len(eval_pairs_restricted)}) ---")
print(f"Popularity (top-10 of the pool):  Hit Rate@10: {pop_hr10_restricted:.4f}")
print(f"Full pipeline:                    Hit Rate@10: {pipeline_hr10_restricted:.4f}   NDCG@10: {pipeline_ndcg10_restricted:.4f}")

eval pairs where true movie is in top-500 popular: 605 / 3427
completed in 380.7s

--- Restricted to top-500-popular candidates (n=605) ---
Popularity (top-10 of the pool):  Hit Rate@10: 0.0744
Full pipeline:                    Hit Rate@10: 0.0760   NDCG@10: 0.0311


In [13]:
# CELL 9 — Save the complete Week 4 test-split evaluation, including the diagnostic

week4_test_results = {
    "test_split": {
        "warm_user_subset_n": len(eval_pairs_warm),
        "full_population_n": len(eval_pairs_full),
    },
    "baselines": {
        "warm_user_subset": {
            "popularity": {"hit_rate_at_10": round(pop_hr10_test_warm, 4), "ndcg_at_10": round(pop_ndcg10_test_warm, 4)},
            "markov": {"hit_rate_at_10": round(markov_hr10_test_warm, 4), "ndcg_at_10": round(markov_ndcg10_test_warm, 4)},
        },
        "full_population": {
            "popularity": {"hit_rate_at_10": round(pop_hr10_test_full, 4), "ndcg_at_10": round(pop_ndcg10_test_full, 4)},
            "markov": {"hit_rate_at_10": round(markov_hr10_test_full, 4), "ndcg_at_10": round(markov_ndcg10_test_full, 4)},
        },
        "markov_fallback_rate": round(fallback_rate_test_warm, 4),
        "val_vs_test_note": (
            "On validation, Markov clearly beat popularity (2.11% vs 1.32% Hit Rate@10, "
            "n=5454). On test (n=3427), this narrowly reversed (0.76% vs 1.31%, a "
            "difference of just 19 hits) -- fallback rate identical (0.8%) across both "
            "splits, ruling out a coverage explanation. Likely sampling variability / "
            "period drift at the smaller test scale rather than a systematic weakness."
        ),
    },
    "full_pipeline": {
        "full_catalog_candidates": {
            "n_users": len(eval_pairs_warm),
            "candidate_pool_size": len(gru_movie_to_idx),
            "hit_rate_at_10": round(0.0123, 4),
            "ndcg_at_10": round(0.0051, 4),
        },
        "restricted_top500_popular_candidates": {
            "n_users": len(eval_pairs_restricted),
            "candidate_pool_size": 500,
            "pct_true_movies_in_pool": round(100 * len(eval_pairs_restricted) / len(eval_pairs_warm), 1),
            "hit_rate_at_10": round(pipeline_hr10_restricted, 4),
            "ndcg_at_10": round(pipeline_ndcg10_restricted, 4),
            "popularity_hit_rate_at_10_same_subset": round(pop_hr10_restricted, 4),
        },
    },
    "finding": (
        "On the full 37,335-candidate test evaluation, the pipeline (1.23% Hit Rate@10) "
        "underperformed popularity (1.31%) despite beating Markov (0.76%). A targeted "
        "diagnostic restricting candidates to the 500 most popular movies (n=605 users "
        "whose true next movie fell in this pool) showed the pipeline performing "
        "comparably to popularity (7.60% vs 7.44%), ruling out an inability to compete "
        "with popular titles specifically. The full-catalog gap is better explained by: "
        "only 17.7% of true next-movies in the test set were themselves top-500-popular; "
        "for the remaining 82.3% (long-tail titles), popularity's baseline advantage in "
        "this narrow single-next-item prediction task outweighs the pipeline's "
        "personalization gains. Suggests the pipeline may be better suited to broader "
        "Top-N recommendation quality than this specific strict-next-item metric. "
        "Future work: popularity-weighted or hard-negative sampling during meta-model "
        "training; alternate evaluation protocols (broader relevance windows, "
        "ranking-based metrics beyond strict Hit Rate@k)."
    ),
}

with open("/kaggle/working/week4_test_results.json", "w") as f:
    json.dump(week4_test_results, f, indent=2)

print(json.dumps(week4_test_results, indent=2))
print("\nsaved: week4_test_results.json (also saved earlier: baseline_test_results.json)")

{
  "test_split": {
    "warm_user_subset_n": 3427,
    "full_population_n": 3602
  },
  "baselines": {
    "warm_user_subset": {
      "popularity": {
        "hit_rate_at_10": 0.0131,
        "ndcg_at_10": 0.0059
      },
      "markov": {
        "hit_rate_at_10": 0.0076,
        "ndcg_at_10": 0.0037
      }
    },
    "full_population": {
      "popularity": {
        "hit_rate_at_10": 0.0155,
        "ndcg_at_10": 0.0072
      },
      "markov": {
        "hit_rate_at_10": 0.0092,
        "ndcg_at_10": 0.0046
      }
    },
    "markov_fallback_rate": 0.0085,
    "val_vs_test_note": "On validation, Markov clearly beat popularity (2.11% vs 1.32% Hit Rate@10, n=5454). On test (n=3427), this narrowly reversed (0.76% vs 1.31%, a difference of just 19 hits) -- fallback rate identical (0.8%) across both splits, ruling out a coverage explanation. Likely sampling variability / period drift at the smaller test scale rather than a systematic weakness."
  },
  "full_pipeline": {
    "full_ca